# ROBUST04 ULTIMATE Pipeline - Target MAP 0.35

## Strategy:
1. **Test both Pure RRF and Weighted RRF with k=10**
2. **Pick the winner**
3. **Add SPLADE as 4th signal**
4. **Reach MAP 0.35!** 🎯

## Parameter Tuning Results (Pure RRF):
- Best k: **10** (not 30, not 60!)
- Best depth: **1000**
- Pure RRF (k=10, d=1000): **MAP 0.2759**
- Original Weighted (k=30, d=300): MAP 0.2738

## This Notebook:
- Phase 1: Test Pure vs Weighted with k=10
- Phase 2: Add SPLADE (expected +0.05-0.07 MAP)
- Phase 3: Final submission → **MAP 0.33-0.35** ✅

In [ ]:
hugging_face_1bLLamaInstruct = "[REDACTED_HF_TOKEN]"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# Java 21 Setup
import os
import subprocess

!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

!java -version

In [ ]:
# Install packages
!pip install -q transformers>=4.46.0 pyserini==0.22.1 ir_measures torch sentence-transformers numpy faiss-cpu --no-cache

In [ ]:
import os, torch, numpy as np
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
from pyserini.encode import SpladeQueryEncoder
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Google Drive Setup
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'
if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
else:
    for path in ['/content/drive/MyDrive/TEXT/FinalProject', '/content/drive/MyDrive/FinalProject', '/content/drive/MyDrive']:
        if os.exists(os.path.join(path, 'queriesROBUST.txt')):
            os.chdir(path)
            break

print(f"Working directory: {os.getcwd()}")

## Configuration

In [ ]:
# ULTIMATE CONFIGURATION
USE_MONOT5 = True
USE_SPLADE = True

# OPTIMAL PARAMETERS FROM TUNING
OPTIMAL_K = 10  # Best from parameter tuning!
OPTIMAL_RERANK_DEPTH = 1000

# Weights for weighted RRF (will test both pure and weighted)
W_BM25 = 1.0
W_RM3 = 1.5
W_SPLADE = 1.8
W_MONOT5 = 2.0

print(f"\n🎯 ULTIMATE Configuration:")
print(f"  • k = {OPTIMAL_K} (from tuning)")
print(f"  • rerank_depth = {OPTIMAL_RERANK_DEPTH}")
print(f"  • Will test: Pure RRF vs Weighted RRF")
print(f"  • Then add: SPLADE (4-way fusion)")
print(f"\n📈 Expected Final: MAP 0.33-0.35")

## Load Index and Searchers

In [ ]:
print("Loading indexes...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 & RM3
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
rm3_searcher.set_rm3(fb_docs=20, fb_terms=15, original_query_weight=0.6)

print("✓ BM25 + RM3 loaded")

# SPLADE
splade_searcher, splade_encoder = None, None
if USE_SPLADE:
    try:
        print("Loading SPLADE (may take a few minutes)...")
        splade_searcher = LuceneSearcher.from_prebuilt_index('beir-v1.0.0-robust04-splade-pp-ed')
        splade_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil')
        print("✓ SPLADE loaded")
    except Exception as e:
        print(f"⚠️ SPLADE failed: {e}")
        USE_SPLADE = False

In [ ]:
# Load queries and qrels
def load_queries(f):
    qs = {}
    with open(f) as file:
        for line in file:
            parts = line.strip().split(None, 1)
            if len(parts) == 2: qs[parts[0]] = parts[1]
    return qs

def load_qrels(f):
    qr = defaultdict(dict)
    with open(f) as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) >= 4: qr[parts[0]][parts[2]] = int(parts[3])
    return qr

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

In [ ]:
# Load MonoT5
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5...")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    try:
        monot5_tokenizer = AutoTokenizer.from_pretrained("castorini/monot5-base-msmarco-10k")
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained("castorini/monot5-base-msmarco-10k").to(device)
        monot5_model.eval()
        print("✓ MonoT5 loaded")
    except Exception as e:
        print(f"MonoT5 failed: {e}")
        USE_MONOT5 = False

## Core Functions

In [ ]:
def retrieve_all(query, k=1000):
    """Get results from all retrievers"""
    bm25_hits = bm25_searcher.search(query, k=k)
    rm3_hits = rm3_searcher.search(query, k=k)
    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]
    
    splade_results = []
    if USE_SPLADE and splade_searcher and splade_encoder:
        try:
            encoded = splade_encoder.encode(query)
            splade_hits = splade_searcher.search(encoded, k=k)
            splade_results = [(h.docid, h.score) for h in splade_hits]
        except: pass
    
    return bm25_results, rm3_results, splade_results

def rerank_monot5(query, doc_ids, batch_size=8):
    if not USE_MONOT5 or monot5_model is None: return None
    
    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc and doc.raw():
                doc_texts.append(doc.raw()[:2000])
                valid_ids.append(did)
        except: pass
    
    if not doc_texts: return None
    
    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]
    true_id = monot5_tokenizer.encode("true")[0]
    false_id = monot5_tokenizer.encode("false")[0]
    
    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            with torch.no_grad():
                outputs = monot5_model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
                logits = outputs.scores[0]
                probs = torch.softmax(torch.stack([logits[:, false_id], logits[:, true_id]], dim=1), dim=1)
                all_scores.extend(probs[:, 1].cpu().tolist())
        except:
            all_scores.extend([0.5] * len(batch))
    
    return dict(zip(valid_ids, all_scores))

## Ultimate Pipeline (Pure + Weighted + SPLADE)

In [ ]:
def ultimate_pipeline(query, mode='weighted', use_splade=True, k_rrf=OPTIMAL_K, rerank_depth=OPTIMAL_RERANK_DEPTH):
    """
    Ultimate RRF Pipeline
    
    Args:
        mode: 'pure' or 'weighted'
        use_splade: Include SPLADE or not
        k_rrf: RRF constant (default: 10)
        rerank_depth: MonoT5 reranking depth (default: 1000)
    """
    # Stage 1: Retrieve
    bm25_results, rm3_results, splade_results = retrieve_all(query)
    
    # Build rankings
    bm25_rank = {d: r for r, (d, _) in enumerate(bm25_results, 1)}
    rm3_rank = {d: r for r, (d, _) in enumerate(rm3_results, 1)}
    splade_rank = {d: r for r, (d, _) in enumerate(splade_results, 1)} if use_splade and splade_results else {}
    
    # Stage 2: MonoT5 Reranking
    monot5_rank = {}
    if USE_MONOT5 and monot5_model:
        top_docs = set()
        top_docs.update([d for d, _ in bm25_results[:rerank_depth]])
        if splade_results and use_splade:
            top_docs.update([d for d, _ in splade_results[:rerank_depth]])
        top_docs = list(top_docs)[:rerank_depth]
        
        scores = rerank_monot5(query, top_docs)
        if scores:
            sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
            monot5_rank = {d: r for r, (d, _) in enumerate(sorted_docs, 1)}
    
    # Stage 3: RRF Fusion
    all_docs = set(bm25_rank.keys()) | set(rm3_rank.keys()) | set(splade_rank.keys())
    rrf_scores = {}
    
    for doc in all_docs:
        score = 0.0
        
        if mode == 'pure':
            # Pure RRF (equal weights)
            if doc in bm25_rank: score += 1.0 / (k_rrf + bm25_rank[doc])
            if doc in rm3_rank: score += 1.0 / (k_rrf + rm3_rank[doc])
            if doc in splade_rank: score += 1.0 / (k_rrf + splade_rank[doc])
            if doc in monot5_rank: score += 1.0 / (k_rrf + monot5_rank[doc])
        else:
            # Weighted RRF
            if doc in bm25_rank: score += W_BM25 * (1.0 / (k_rrf + bm25_rank[doc]))
            if doc in rm3_rank: score += W_RM3 * (1.0 / (k_rrf + rm3_rank[doc]))
            if doc in splade_rank: score += W_SPLADE * (1.0 / (k_rrf + splade_rank[doc]))
            if doc in monot5_rank: score += W_MONOT5 * (1.0 / (k_rrf + monot5_rank[doc]))
        
        rrf_scores[doc] = score
    
    # Sort and return
    final = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(str(d), float(s), int(r)) for r, (d, s) in enumerate(final[:1000], 1)]

## Phase 1: Test Pure vs Weighted (k=10, no SPLADE)

In [ ]:
import time

print("="*70)
print("🧪 PHASE 1: PURE vs WEIGHTED RRF (k=10, depth=1000)")
print("="*70)
print("Testing both on training set (without SPLADE first)...\n")

def evaluate_config(mode_name, pipeline_mode):
    print(f"\n{'='*70}")
    print(f"Testing: {mode_name}")
    print(f"{'='*70}")
    
    results = []
    start = time.time()
    
    for i, qid in enumerate(train_qids, 1):
        if i % 10 == 0: print(f"  [{i}/{len(train_qids)}] Processing...")
        
        try:
            res = ultimate_pipeline(queries[qid], mode=pipeline_mode, use_splade=False)
            for docid, score, rank in res:
                results.append(ir_measures.ScoredDoc(query_id=str(qid), doc_id=str(docid), score=float(score)))
        except Exception as e:
            print(f"  Error on {qid}: {str(e)[:50]}")
    
    elapsed = time.time() - start
    
    # Evaluate
    qrels_list = [ir_measures.Qrel(query_id=str(q), doc_id=str(d), relevance=int(r)) 
                  for q, docs in qrels.items() for d, r in docs.items()]
    
    metrics = ir_measures.calc_aggregate([MAP, nDCG@10, P@10], qrels_list, results)
    
    print(f"\n  Results: MAP={metrics[MAP]:.4f}, nDCG@10={metrics[nDCG@10]:.4f}, P@10={metrics[P@10]:.4f}")
    print(f"  Time: {elapsed:.1f}s")
    
    return metrics[MAP]

# Test both
pure_map = evaluate_config("Pure RRF (k=10, no weights)", "pure")
weighted_map = evaluate_config("Weighted RRF (k=10, W_BM25=1.0, W_RM3=1.5, W_MONOT5=2.0)", "weighted")

print("\n" + "="*70)
print("📊 PHASE 1 RESULTS")
print("="*70)
print(f"Pure RRF:     MAP = {pure_map:.4f}")
print(f"Weighted RRF: MAP = {weighted_map:.4f}")

winner_mode = "weighted" if weighted_map > pure_map else "pure"
winner_map = max(pure_map, weighted_map)

print(f"\n🏆 Winner: {'Weighted' if winner_mode == 'weighted' else 'Pure'} RRF (MAP = {winner_map:.4f})")
print(f"\nProceeding to Phase 2 with: {winner_mode.upper()} RRF")
print("="*70)

## Phase 2: Add SPLADE to Winner

In [ ]:
print("="*70)
print(f"🚀 PHASE 2: ADDING SPLADE TO {winner_mode.upper()} RRF")
print("="*70)
print(f"Testing 4-way fusion on training set...\n")

if USE_SPLADE:
    final_map = evaluate_config(f"{winner_mode.capitalize()} RRF + SPLADE (4-way)", winner_mode)
    
    print("\n" + "="*70)
    print("📊 PHASE 2 RESULTS")
    print("="*70)
    print(f"Baseline ({winner_mode}, 3-way): MAP = {winner_map:.4f}")
    print(f"With SPLADE (4-way):            MAP = {final_map:.4f}")
    improvement = final_map - winner_map
    print(f"\nSPLADE Boost: {improvement:+.4f} ({improvement/winner_map*100:+.2f}%)")
    
    if final_map >= 0.32:
        print(f"\n🎉 EXCELLENT! MAP {final_map:.4f} is in competitive range (0.32+)!")
    elif final_map >= 0.30:
        print(f"\n✓ GOOD! MAP {final_map:.4f}. Close to target!")
    else:
        print(f"\n⚠️ MAP {final_map:.4f}. May need weight tuning.")
    
    print("="*70)
else:
    print("⚠️ SPLADE not available. Using 3-way fusion only.")
    final_map = winner_map

## Phase 3: Generate Final Test Results

In [ ]:
print("="*70)
print("🏁 PHASE 3: GENERATING FINAL TEST RESULTS")
print("="*70)
print(f"Configuration:")
print(f"  • Mode: {winner_mode.upper()} RRF")
print(f"  • k = {OPTIMAL_K}")
print(f"  • rerank_depth = {OPTIMAL_RERANK_DEPTH}")
print(f"  • SPLADE: {'YES (4-way)' if USE_SPLADE else 'NO (3-way)'}")
print(f"  • Test queries: {len(test_qids)}")
print(f"\n🎯 Expected MAP: ~{final_map:.3f} (based on training)")
print("="*70)
print()

test_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    print(f"[{i}/{len(test_qids)}] Query {qid}: {query_text[:60]}...")
    
    try:
        results = ultimate_pipeline(query_text, mode=winner_mode, use_splade=USE_SPLADE)
        
        for docid, score, rank in results:
            test_results.append({'qid': str(qid), 'docid': str(docid), 'rank': int(rank), 'score': float(score)})
        
        print(f"  ✓ {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:50]}")
    
    if i % 10 == 0:
        elapsed = time.time() - start_time
        remaining = (len(test_qids) - i) * (elapsed / i)
        print(f"\n  Progress: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f}min, Remaining: {remaining/60:.1f}min\n")

total_time = time.time() - start_time

print("\n" + "="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(test_results):,}")
print(f"  Time: {total_time/60:.1f} minutes")
print("="*70)

## Save Final Submission

In [ ]:
filename = f"run_3_ultimate_{winner_mode}_{('splade' if USE_SPLADE else 'no_splade')}.res"

with open(filename, 'w') as f:
    for r in test_results:
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {r['score']:.6f} ultimate\n")

print(f"✓ Saved to: {filename}")
print("\nFirst 5 lines:")
with open(filename) as f:
    for i, line in enumerate(f):
        if i < 5: print(line.strip())
        else: break

print("\n" + "="*70)
print("🎉 ULTIMATE SUBMISSION READY!")
print("="*70)
print(f"File: {filename}")
print("\n📊 Final Configuration:")
print(f"  • RRF Mode: {winner_mode.upper()}")
print(f"  • k = {OPTIMAL_K}")
print(f"  • rerank_depth = {OPTIMAL_RERANK_DEPTH}")
print(f"  • Signals: BM25 + RM3 + {'SPLADE + ' if USE_SPLADE else ''}MonoT5")
print(f"\n🎯 Training MAP: {final_map:.4f}")
print(f"   Expected Test MAP: ~{final_map:.3f}")
if final_map >= 0.32:
    print(f"\n🌟 Target 0.35: Within reach!")
print("="*70)